# FinBERT Model Training Notebook

This notebook is designed to run on Talapas with a GPU. It imports the same project code used by `train_model.py`, so the notebook and script stay consistent.


In [ ]:
import os
import json
from pathlib import Path

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split

from earnings_transcript_predictor.dataset import EarningsDataset, load_records
from earnings_transcript_predictor.models import EarningsModel
from train_model import set_seed, train_epoch, eval_epoch


## Configuration

Change `SEED` to test how sensitive the model is to different train/test splits.


In [ ]:
DATA_DIR = Path('/gpfs/home/smaccall/earnings-call-prediction/notebooks/data/transcripts')
if not DATA_DIR.exists():
    DATA_DIR = Path('notebooks/data/transcripts')

OUTPUT_DIR = Path('checkpoints')
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
BATCH_SIZE = 8
HEAD_EPOCHS = 2
FULL_EPOCHS = 3
HEAD_LR = 1e-3
FULL_LR = 2e-5

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Data directory: {DATA_DIR}')


## Load Dataset


In [ ]:
records = load_records(DATA_DIR)
dataset = EarningsDataset(records, max_length=512)

n_test = max(1, int(len(dataset) * 0.2))
n_train = len(dataset) - n_test
generator = torch.Generator().manual_seed(SEED)
train_ds, test_ds = random_split(dataset, [n_train, n_test], generator=generator)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
print(f'Train: {n_train} | Test: {n_test}')


## Model

FinBERT provides the transcript representation. A linear regression head predicts the 3-day post-earnings return percentage.


In [ ]:
model = EarningsModel().to(device)
criterion = nn.MSELoss()
history = []


## Phase 1: Frozen FinBERT

Only the regression head trains in this phase. This reduces overfitting risk with a very small dataset.


In [ ]:
model.freeze_bert()
optimizer = torch.optim.AdamW(model.regressor.parameters(), lr=HEAD_LR)

for epoch in range(HEAD_EPOCHS):
    loss = train_epoch(model, train_loader, optimizer, device, criterion)
    rmse, dir_acc = eval_epoch(model, test_loader, device, criterion, n_test)
    history.append({'phase': 'head_only', 'epoch': epoch + 1, 'loss': loss, 'rmse': rmse, 'dir_acc': dir_acc})
    print(f'Head epoch {epoch+1}/{HEAD_EPOCHS} | loss={loss:.4f} | rmse={rmse:.4f} | dir_acc={dir_acc:.2f}')


## Phase 2: Full Fine-Tuning


In [ ]:
model.unfreeze_bert()
optimizer = torch.optim.AdamW(model.parameters(), lr=FULL_LR, weight_decay=0.01)
best_rmse = float('inf')

for epoch in range(FULL_EPOCHS):
    loss = train_epoch(model, train_loader, optimizer, device, criterion)
    rmse, dir_acc = eval_epoch(model, test_loader, device, criterion, n_test)
    history.append({'phase': 'full_finetune', 'epoch': epoch + 1, 'loss': loss, 'rmse': rmse, 'dir_acc': dir_acc})
    print(f'Full epoch {epoch+1}/{FULL_EPOCHS} | loss={loss:.4f} | rmse={rmse:.4f} | dir_acc={dir_acc:.2f}')

    if rmse < best_rmse:
        best_rmse = rmse
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pt')
        print(f'Saved best model with RMSE={best_rmse:.4f}')

with open(OUTPUT_DIR / 'training_history.json', 'w') as f:
    json.dump({'seed': SEED, 'n_train': n_train, 'n_test': n_test, 'history': history, 'best_rmse': best_rmse}, f, indent=2)


## Training Curves


In [ ]:
epochs = list(range(1, len(history) + 1))
losses = [row['loss'] for row in history]
rmses = [row['rmse'] for row in history]
dir_accs = [row['dir_acc'] for row in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, losses, marker='o')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')

axes[1].plot(epochs, rmses, marker='o', color='orange')
axes[1].set_title('Validation RMSE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Return %')

axes[2].plot(epochs, dir_accs, marker='o', color='green')
axes[2].axhline(0.5, color='red', linestyle='--', label='Random baseline')
axes[2].set_title('Directional Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].set_ylim(0, 1)
axes[2].legend()

plt.tight_layout()
plt.show()


## Note on Interpretation

With 148 transcripts and about 29 test examples, a single train/test split is not enough to prove generalization. Run multiple seeds and report the range, not only the best run.
